# Notebook 03: Decision Trees and Random Forests

## Why This Matters

Decision trees are the building blocks of the most powerful tabular ML methods (Random Forests, XGBoost, LightGBM) and remain widely deployed due to interpretability. Understanding WHY a single tree overfits and HOW bagging fixes this is foundational for all ensemble methods. Random forests were state-of-the-art for tabular data before gradient boosting and are still competitive, faster to train, and more robust to hyperparameter choices than GBMs.

In [ ]:
%pip install -q scikit-learn numpy pandas matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.datasets import make_classification, load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score
import warnings; warnings.filterwarnings("ignore")
np.random.seed(42)
print("Ready.")

## 1. Decision Tree Splitting: Gini vs Information Gain

**Gini Impurity**: `Gini = 1 - sum(p_i^2)`
- Range: 0 (pure) to 0.5 (50/50 split for binary)
- Computationally faster than entropy

**Information Gain**: `IG = H(parent) - sum(|child|/|parent| * H(child))`
- Where H = -sum(p_i * log2(p_i)) is Shannon entropy

**Which to use**: In practice they usually select the same splits. Gini is sklearn's default and faster.

**The splitting algorithm**: For each feature, sort unique values and try all midpoints as thresholds. Select feature+threshold with highest information gain. O(n * p * log n) per level.

In [ ]:
class SimpleDecisionTree:
    class Node:
        def __init__(self):
            self.feature = None; self.threshold = None
            self.left = None; self.right = None; self.prediction = None
    
    def __init__(self, max_depth=3, min_samples_split=2):
        self.max_depth = max_depth; self.min_samples_split = min_samples_split
    
    def _gini(self, y):
        if len(y) == 0: return 0.0
        probs = np.unique(y, return_counts=True)[1] / len(y)
        return 1 - np.sum(probs**2)
    
    def _best_split(self, X, y):
        n, p = X.shape; best_gain = -np.inf; best_feat, best_thresh = None, None
        parent_gini = self._gini(y)
        for feat in range(p):
            for thresh in np.unique(X[:, feat])[:-1]:
                left = y[X[:, feat] <= thresh]; right = y[X[:, feat] > thresh]
                if not len(left) or not len(right): continue
                gain = parent_gini - (len(left)/n*self._gini(left) + len(right)/n*self._gini(right))
                if gain > best_gain: best_gain = gain; best_feat = feat; best_thresh = thresh
        return best_feat, best_thresh
    
    def _build(self, X, y, depth):
        node = self.Node()
        if depth >= self.max_depth or len(y) < self.min_samples_split or len(np.unique(y)) == 1:
            node.prediction = np.bincount(y).argmax(); return node
        feat, thresh = self._best_split(X, y)
        if feat is None: node.prediction = np.bincount(y).argmax(); return node
        node.feature = feat; node.threshold = thresh
        mask = X[:, feat] <= thresh
        node.left = self._build(X[mask], y[mask], depth+1)
        node.right = self._build(X[~mask], y[~mask], depth+1)
        return node
    
    def fit(self, X, y):
        self.root = self._build(X, y.astype(int), 0); return self
    
    def _predict_one(self, x, node):
        if node.prediction is not None: return node.prediction
        return self._predict_one(x, node.left if x[node.feature] <= node.threshold else node.right)
    
    def predict(self, X): return np.array([self._predict_one(x, self.root) for x in X])

data = load_breast_cancer()
X, y = data.data[:, :5], data.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
tree_scratch = SimpleDecisionTree(max_depth=4).fit(X_tr, y_tr)
tree_sklearn = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_tr, y_tr)
print(f"From-scratch accuracy: {accuracy_score(y_te, tree_scratch.predict(X_te)):.4f}")
print(f"sklearn accuracy:      {accuracy_score(y_te, tree_sklearn.predict(X_te)):.4f}")

## 2. Why Single Trees Overfit: Bias-Variance

**High variance**: A small change in training data can cause a completely different feature to be selected at the root, leading to a structurally different tree. This is why a single tree's test accuracy can vary dramatically across random splits.

**Bias-variance knobs**:
- `max_depth`: primary control (depths 3-15 typically optimal)
- `min_samples_leaf`: minimum samples per leaf; prevents very specific leaf rules
- `min_samples_split`: minimum samples to split a node

In [ ]:
X, y = make_classification(n_samples=2000, n_features=20, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)

depths = [1, 2, 3, 5, 8, 12, None]
print(f"{"max_depth":<12} {"Train Acc":>10} {"Test Acc":>10} {"CV Mean":>10} {"CV Std":>8}")
for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    cv = cross_val_score(dt, X, y, cv=10)
    dt.fit(X_tr, y_tr)
    train_acc = accuracy_score(y_tr, dt.predict(X_tr))
    test_acc = accuracy_score(y_te, dt.predict(X_te))
    print(f"{str(d):<12} {train_acc:>10.4f} {test_acc:>10.4f} {cv.mean():>10.4f} {cv.std():>8.4f}")

## 3. Random Forests: Why Feature Randomness Helps

**Bagging**: train many trees on bootstrap samples, average predictions -> reduces variance.

**Why feature randomness is critical**: Without it, different bootstrapped trees still tend to select the same strong features at the root. Correlated models don't reduce variance much when averaged. Feature randomness (sqrt(p) features per split) decorrelates the trees - each tree is a 'different view' of the problem.

**Theorem**: ensemble error <= (correlation x variance) / n_estimators. Lower correlation between trees directly reduces ensemble error.

In [ ]:
n_trees = 100
models = {
    "Single Tree": DecisionTreeClassifier(random_state=42),
    "Bagging (no feat randomness)": BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=42), n_estimators=n_trees, max_features=1.0, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=n_trees, random_state=42),
}
print("=== Variance Reduction Across Methods (20-fold CV) ===")
print(f"{"Model":<45} {"Mean Acc":>10} {"Std Acc":>10}")
for name, model in models.items():
    cv = cross_val_score(model, X, y, cv=20)
    print(f"{name:<45} {cv.mean():>10.4f} {cv.std():>10.4f}")
print("\nLower std = lower variance. RF has lowest std because feature randomness decorrelates trees.")

## 4. Feature Importance: MDI vs Permutation

**Mean Decrease Impurity (MDI)**: average Gini reduction from splits on each feature. Fast but biased toward high-cardinality features.

**Permutation Importance**: shuffle each feature's values and measure accuracy drop. Model-agnostic, unbiased, but 10-50x slower. Use when MDI importances look suspicious.

In [ ]:
X, y = make_classification(n_samples=2000, n_features=15, n_informative=5, n_redundant=3, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_tr, y_tr)
mdi_imp = rf.feature_importances_
perm = permutation_importance(rf, X_te, y_te, n_repeats=5, random_state=42)

feat_names = [f"f{i}" for i in range(15)]
colors = ["red" if i < 5 else "steelblue" for i in range(15)]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
idx_mdi = np.argsort(mdi_imp)[::-1]; ax1.bar(np.array(feat_names)[idx_mdi], mdi_imp[idx_mdi], color=np.array(colors)[idx_mdi])
ax1.set_title("MDI (may be biased toward high-cardinality)"); ax1.tick_params(axis="x", rotation=45)
idx_perm = np.argsort(perm.importances_mean)[::-1]; ax2.bar(np.array(feat_names)[idx_perm], perm.importances_mean[idx_perm], color=np.array(colors)[idx_perm])
ax2.set_title("Permutation Importance (unbiased)"); ax2.tick_params(axis="x", rotation=45)
plt.suptitle("Red = truly informative features (f0-f4)")
plt.tight_layout(); plt.savefig("/tmp/feat_imp.png", dpi=80); plt.show()
print(f"RF test accuracy: {accuracy_score(y_te, rf.predict(X_te)):.4f}")

## Summary

### Key Concepts

| Concept | Key Point |
|---|---|
| Gini impurity | 1 - sum(p_i^2); range [0, 0.5] for binary; faster than entropy |
| High variance in trees | Small data change -> completely different tree structure |
| Bagging | Bootstrap + average -> reduces variance |
| Feature randomness | sqrt(p) features per split -> decorrelates trees -> better variance reduction |
| Out-of-bag error | Free validation from bootstrap samples not used in training |
| MDI vs permutation | MDI: fast but biased; Permutation: slow but unbiased |

### Common Pitfalls
- Using a single deep tree without cross-validation
- Setting n_estimators too low (200+ for stable importance)
- Using MDI importance with high-cardinality features
- Not tuning min_samples_leaf (strong regularizer often overlooked)
